# Installation des Conda Environments 
(nur für Windows auf den Laborrechnern)

Führe das **im PowerShell-Terminal** aus:

### Fehlermeldungn
⚠️ **Hinweis (PowerShell): nicht im Notebook ausführen**
> "Running cells with 'base (Python 3.13.11)' requires the ipykernel package.
> Create a Python Environment with the required packages.
> Or install 'ipykernel' using the command: 'conda install -n base ipykernel --update-deps --force-reinstall'"

**Environment wurde nicht gefunden** (Powershell ist nicht im Projektordner).

> environment.yml nicht gefunden. Bitte in den Projektordner (oder Unterordner) wechseln und erneut ausführen.
> In Zeile:2 Zeichen:5
> +     throw "environment.yml nicht gefunden. Bitte in den Projektordner ..."
> +     ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
>     + CategoryInfo          : OperationStopped: (environment.yml...neut ausführen.:String) [], RuntimeException
>     + FullyQualifiedErrorId : environment.yml nicht gefunden. Bitte in den Projektordner (oder Unterordner) wechseln und erneut ausführen.

✅ **Lösung:**
> Wechsle zuerst in den Projektordner:  
> `cd H:\........` # Hier den Pfad vom Projekt kopieren
>
> Alternativ im Explorer im Projektordner: **Shift + Rechtsklick** → **„PowerShell-Fenster hier öffnen“**.
>
> Kopiere den Installationsblock und führe ihn erneut aus.

In [ ]:
$condaCmd = Get-Command conda -ErrorAction SilentlyContinue
if ($condaCmd) {
    $conda = $condaCmd.Source
} else {
    $candidates = @(
        "$env:USERPROFILE\miniconda3\Scripts\conda.exe",
        "$env:USERPROFILE\anaconda3\Scripts\conda.exe",
        "C:\ProgramData\miniconda3\Scripts\conda.exe",
        "C:\ProgramData\Anaconda3\Scripts\conda.exe"
    )
    $conda = $candidates | Where-Object { Test-Path $_ } | Select-Object -First 1
}

if (-not $conda) { throw "conda.exe nicht gefunden." }

# Projektordner automatisch finden (sucht nach environment.yml ab aktuellem Ordner nach oben)
$dir = (Get-Location).Path
while ($dir -and -not (Test-Path (Join-Path $dir "environment.yml"))) {
    $parent = Split-Path $dir -Parent
    if ($parent -eq $dir) { $dir = $null } else { $dir = $parent }
}

if (-not $dir) {
    throw "environment.yml nicht gefunden. Bitte in den Projektordner (oder Unterordner) wechseln und erneut ausführen."
}

$projectRoot = $dir
Set-Location $projectRoot

# 1) Kurs-Umgebung im User-Bereich erstellen/aktualisieren (ohne Admin-Rechte)
$envPath = "$env:USERPROFILE\.conda\envs\environment_it_energy"
& $conda env create --prefix $envPath -f .\environment.yml
if ($LASTEXITCODE -ne 0) {
    & $conda env update --prefix $envPath -f .\environment.yml --prune
    if ($LASTEXITCODE -ne 0) { throw "Umgebung konnte nicht erstellt/aktualisiert werden." }
}

# 2) Kernel für die neue Umgebung registrieren
& $conda run -p $envPath python -m ipykernel install --user --name environment_it_energy --display-name "Python (environment_it_energy)"
if ($LASTEXITCODE -ne 0) { throw "Kernel-Registrierung fehlgeschlagen." }

Write-Host "Fertig. Wähle jetzt im Notebook den Kernel: Python (environment_it_energy)"

: 

# Installationstest

Führe alle Zellen aus. Wenn am Ende **✅ Alle Tests bestanden!** erscheint, ist deine Umgebung korrekt eingerichtet.

In [1]:
import sys
print(f"Python-Version: {sys.version}")
print(f"Python-Pfad:    {sys.executable}")
print()

# Pakete importieren und Versionen prüfen
pakete = {}
fehler = []

try:
    import pandas as pd
    pakete["pandas"] = pd.__version__
except ImportError as e:
    fehler.append(f"pandas: {e}")

try:
    import numpy as np
    pakete["numpy"] = np.__version__
except ImportError as e:
    fehler.append(f"numpy: {e}")

try:
    import matplotlib
    pakete["matplotlib"] = matplotlib.__version__
except ImportError as e:
    fehler.append(f"matplotlib: {e}")

try:
    import plotly
    pakete["plotly"] = plotly.__version__
except ImportError as e:
    fehler.append(f"plotly: {e}")

try:
    import sklearn
    pakete["scikit-learn"] = sklearn.__version__
except ImportError as e:
    fehler.append(f"scikit-learn: {e}")

try:
    import pypsa
    pakete["pypsa"] = pypsa.__version__
except ImportError as e:
    fehler.append(f"pypsa: {e}")

try:
    import linopy
    pakete["linopy"] = linopy.__version__
except ImportError as e:
    fehler.append(f"linopy: {e}")

try:
    import highspy
    pakete["highspy"] = highspy.HIGHS_VERSION_MAJOR
    pakete["highspy"] = "installiert"
except ImportError as e:
    fehler.append(f"highspy: {e}")

try:
    import hydra
    pakete["hydra-core"] = hydra.__version__
except ImportError as e:
    fehler.append(f"hydra-core: {e}")

try:
    import openpyxl
    pakete["openpyxl"] = openpyxl.__version__
except ImportError as e:
    fehler.append(f"openpyxl: {e}")

# Ergebnis
print("=" * 50)
print("INSTALLATIONSTEST")
print("=" * 50)

for name, version in pakete.items():
    print(f"  ✅ {name:20s} {version}")

if fehler:
    print()
    for f in fehler:
        print(f"  ❌ {f}")
    print()
    print("⚠️  Einige Pakete fehlen! Bitte environment.yml erneut installieren:")
    print("    conda env update -f environment.yml --prune")
else:
    print()
    print("✅ Alle Tests bestanden! Die Umgebung ist korrekt eingerichtet.")

# %%
# Solver-Test: Kann HiGHS ein einfaches Problem lösen?
print("\n" + "=" * 50)
print("SOLVER-TEST (HiGHS)")
print("=" * 50)

try:
    import linopy
    m = linopy.Model()
    x = m.add_variables(lower=0, name="x")
    y = m.add_variables(lower=0, name="y")
    m.add_constraints(x + y <= 10, name="c1")
    m.add_objective(2 * x + 3 * y, sense="max")
    m.solve(solver_name="highs")

    if m.status == "ok":
        print(f"  ✅ HiGHS Solver funktioniert!")
        print(f"     Ergebnis: x={x.solution.values:.1f}, y={y.solution.values:.1f}")
        print(f"     Zielfunktion: {m.objective.value:.1f} (erwartet: 30.0)")
    else:
        print(f"  ⚠️  Solver-Status: {m.status}")
except Exception as e:
    print(f"  ❌ Solver-Fehler: {e}")
    print("     Versuche alternativ: conda install -c conda-forge glpk")

# %%
# Plotly-Test: Kann ein interaktiver Plot erzeugt werden?
print("\n" + "=" * 50)
print("PLOTLY-TEST")
print("=" * 50)

try:
    import plotly.graph_objects as go
    fig = go.Figure(data=go.Scatter(x=[1, 2, 3], y=[1, 3, 2], mode="lines+markers"))
    fig.update_layout(title="Testplot – wenn du das siehst, funktioniert Plotly! ✅")
    fig.show()
    print("  ✅ Plotly funktioniert!")
except Exception as e:
    print(f"  ❌ Plotly-Fehler: {e}")


Python-Version: 3.11.15 | packaged by conda-forge | (main, Mar  5 2026, 16:36:00) [MSC v.1944 64 bit (AMD64)]
Python-Pfad:    c:\Users\hswt141mar\.conda\envs\environment_it_energy\python.exe

INSTALLATIONSTEST
  ✅ pandas               3.0.2
  ✅ numpy                2.4.3
  ✅ matplotlib           3.10.8
  ✅ plotly               6.6.0
  ✅ scikit-learn         1.8.0
  ✅ pypsa                1.0.6
  ✅ linopy               0.6.6
  ✅ highspy              installiert
  ✅ hydra-core           1.3.2
  ✅ openpyxl             3.1.5

✅ Alle Tests bestanden! Die Umgebung ist korrekt eingerichtet.

SOLVER-TEST (HiGHS)
  ✅ HiGHS Solver funktioniert!
     Ergebnis: x=0.0, y=10.0
     Zielfunktion: 30.0 (erwartet: 30.0)

PLOTLY-TEST


  ✅ Plotly funktioniert!


## Nächster Schritt

Wenn alle Tests bestanden sind, öffne **`0_Wiederholung_Python`** und
beginne mit der ersten Übung! 🎉